# AI-Based Climate Change Article Classification System

## Section 1: Project Introduction

**Project title:** AI-Based Climate Change Article Classification System

**Objective:** Build an NLP and Machine Learning system that accepts an English article or paragraph and classifies it into one of **six** selected categories. The system outputs a probability (percentage) for every category and reports the highest-probability category as the final prediction.

**Dataset:** `SciDCC.csv` (a collection of science-news articles from ScienceDaily). Columns: `Date`, `Link`, `Title`, `Summary`, `Body`, `Category`, `Year`.

**Six selected categories (and no others):**
1. Climate
2. Pollution
3. Environment
4. Weather
5. Ozone Holes
6. Agriculture & Food

**NLP classification:** the task is *supervised multi-class text classification*. Raw text is converted into numeric TF-IDF features, and a classifier learns to map those features to one of the six categories.

**Machine Learning workflow:**
load data -> explore -> filter six categories -> clean text -> EDA -> train/test split (80/20, stratified) -> TF-IDF + classifier `Pipeline`s -> 5-fold stratified cross-validation on the training set -> select the best model by mean CV Macro F1 -> evaluate **once** on the untouched test set -> save models and reports.

**Expected input:** an English article or paragraph.  
**Expected output:** a probability for each of the six categories, and the category with the highest probability as the final prediction.

## Section 2: Import Libraries

In [1]:
import os
import re
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import display

from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix)

try:
    from wordcloud import WordCloud
    WORDCLOUD_AVAILABLE = True
except ImportError:
    WORDCLOUD_AVAILABLE = False

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)
RANDOM_STATE = 42
print("wordcloud package available:", WORDCLOUD_AVAILABLE)

wordcloud package available: True


## Section 3: Load Dataset

All paths are relative to the project root, which is located automatically, so the notebook works when opened from the `notebooks/` folder (or from the project root).

In [2]:
def find_project_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "data" / "SciDCC.csv").exists():
            return p
    raise FileNotFoundError("Could not find data/SciDCC.csv in the current folder or any parent folder.")

ROOT = find_project_root()
DATA_DIR, MODELS_DIR = ROOT / "data", ROOT / "models"
REPORTS_DIR, IMAGES_DIR = ROOT / "reports", ROOT / "images"
for d in (MODELS_DIR, REPORTS_DIR, IMAGES_DIR):
    d.mkdir(exist_ok=True)

df = pd.read_csv(DATA_DIR / "SciDCC.csv")

# Safe handling of column-name capitalization / whitespace
canonical = ["Date", "Link", "Title", "Summary", "Body", "Category", "Year"]
lookup = {c.lower(): c for c in canonical}
df.columns = [lookup.get(str(c).strip().lower(), str(c).strip()) for c in df.columns]
print("Project root:", ROOT.name)
print("Columns after normalisation:", list(df.columns))

FileNotFoundError: Could not find data/SciDCC.csv in the current folder or any parent folder.

In [ ]:
print("First five records:")
display(df.head())

In [ ]:
print("Last five records:")
display(df.tail())

In [ ]:
print("Dataset shape:", df.shape)
print("Column names:", list(df.columns))
print()
print("Data types:")
print(df.dtypes)
print()
df.info()

## Section 4: Dataset Exploration

In [ ]:
if "Category" in df.columns:
    df["Category"] = df["Category"].astype(str).str.strip()

print("Total records:", len(df))
print()
print("Missing values per column:")
print(df.isna().sum())
print()
print("Fully duplicated rows:", int(df.duplicated().sum()))
print("Duplicated Body values (non-null):", int(df["Body"].dropna().duplicated().sum()))

In [ ]:
original_counts = df["Category"].value_counts()
print("Number of original categories:", df["Category"].nunique())
print()
print("Original category counts:")
display(original_counts.rename("records").to_frame())

fig, ax = plt.subplots(figsize=(10, 7))
colors = ["#2E7D32" if c in ["Climate", "Pollution", "Environment", "Weather", "Ozone Holes", "Agriculture & Food"] else "#B0BEC5"
          for c in original_counts.index]
ax.barh(original_counts.index[::-1], original_counts.values[::-1], color=colors[::-1])
ax.set_xlabel("Number of articles")
ax.set_title("Original SciDCC category distribution (six selected categories highlighted)")
plt.tight_layout()
fig.savefig(IMAGES_DIR / "original_category_distribution.png", dpi=150)
plt.show()

## Section 5: Filter Six Categories

Only the six project categories are kept. All other categories are removed. Category names are used exactly as they appear in the dataset (no renaming).

In [ ]:
SELECTED = ["Climate", "Pollution", "Environment", "Weather", "Ozone Holes", "Agriculture & Food"]
EXPECTED = {"Climate": 700, "Pollution": 945, "Environment": 478, "Weather": 719,
            "Ozone Holes": 623, "Agriculture & Food": 844}

missing_cats = [c for c in SELECTED if c not in set(df["Category"])]
assert not missing_cats, f"Selected categories not found in the dataset: {missing_cats}"

df_six = df[df["Category"].isin(SELECTED)].copy().reset_index(drop=True)
actual = df_six["Category"].value_counts().reindex(SELECTED)

cmp = pd.DataFrame({"Expected": pd.Series(EXPECTED), "Actual": actual})
cmp["Difference"] = cmp["Actual"] - cmp["Expected"]
cmp.loc["Total"] = [cmp["Expected"].sum(), cmp["Actual"].sum(), cmp["Difference"].sum()]
display(cmp.astype(int))
print("Actual counts match the expected counts:", bool((cmp["Difference"] == 0).all()))

print("Filtered dataset shape:", df_six.shape)
print("Unique categories remaining:", df_six["Category"].nunique())
assert df_six["Category"].nunique() == 6 and set(df_six["Category"]) == set(SELECTED)
print("Verified: exactly six unique categories remain.")

df_six.to_csv(DATA_DIR / "filtered_SciDCC.csv", index=False)
print("Saved:", (DATA_DIR / "filtered_SciDCC.csv").relative_to(ROOT))

## Section 6: Data Cleaning and Preprocessing

Rules used:
- `Body` is the primary article text. If `Body` is missing or empty, `Summary` is used instead.
- `Title` is added in front of the text when available.
- `Date`, `Link`, `Year` and `Category` are **never** used as input features. The target is `Category`.
- Cleaning: lower-casing, removal of URLs and e-mail addresses, removal of punctuation/special characters, whitespace normalisation. Letters **and digits** are kept so terms such as *CO2* or *O3* survive. Stop words are **not** removed here (TF-IDF handles English stop words), so meaningful climate words are never removed by hand.
- Empty texts are dropped and duplicate articles are removed.

In [ ]:
def clean_text(text):
    # NOTE: app.py contains an identical copy of this function.
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

data = df_six.copy()
for col in ["Title", "Summary", "Body"]:
    data[col] = data[col].fillna("").astype(str).str.strip()

n_body_empty = int((data["Body"] == "").sum())
data["article_text"] = np.where(data["Body"] != "", data["Body"], data["Summary"])
data["article_text"] = (data["Title"] + ". " + data["article_text"]).str.strip(". ").str.strip()
data["clean_text"] = data["article_text"].apply(clean_text)

before = len(data)
empty_mask = data["clean_text"] == ""
data = data[~empty_mask]
n_empty_removed = before - len(data)

before = len(data)
data = data.drop_duplicates(subset="clean_text").reset_index(drop=True)
n_dup_removed = before - len(data)

print("Records with empty Body (Summary used instead):", n_body_empty)
print("Records removed for empty text:", n_empty_removed)
print("Duplicate articles removed:", n_dup_removed)
print("Records available for modelling:", len(data))
print()
print("Example (raw text, first 300 chars):")
print(data["article_text"].iloc[0][:300])
print()
print("Example (cleaned text, first 300 chars):")
print(data["clean_text"].iloc[0][:300])
display(data["Category"].value_counts().reindex(SELECTED).rename("records after cleaning").to_frame())

## Section 7: Exploratory Data Analysis

In [ ]:
cat_counts = data["Category"].value_counts().reindex(SELECTED)
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=cat_counts.values, y=cat_counts.index, palette="viridis", ax=ax)
for i, v in enumerate(cat_counts.values):
    ax.text(v + 5, i, str(v), va="center")
ax.set_xlabel("Number of articles")
ax.set_ylabel("")
ax.set_title("Category distribution (six selected categories)")
plt.tight_layout()
fig.savefig(IMAGES_DIR / "category_distribution.png", dpi=150)
plt.show()

In [ ]:
data["word_count"] = data["clean_text"].str.split().str.len()
data["char_count"] = data["clean_text"].str.len()
print("Word count summary (all selected articles):")
display(data["word_count"].describe().round(1).to_frame().T)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data["word_count"], bins=40, kde=True, color="#2E7D32", ax=axes[0])
axes[0].set_title("Article length distribution (words)")
axes[0].set_xlabel("Words per article")
sns.boxplot(data=data, x="word_count", y="Category", order=SELECTED, palette="viridis", ax=axes[1])
axes[1].set_title("Article length by category")
axes[1].set_xlabel("Words per article")
axes[1].set_ylabel("")
plt.tight_layout()
fig.savefig(IMAGES_DIR / "article_length_distribution.png", dpi=150)
plt.show()

In [ ]:
# Category-wise article analysis
cat_stats = data.groupby("Category")["word_count"].agg(["count", "mean", "median", "min", "max"]).reindex(SELECTED).round(1)
cat_stats.columns = ["articles", "mean words", "median words", "min words", "max words"]
display(cat_stats)

# Most frequent words (visualisation only; these extra words are NOT removed from the model features)
VIS_STOP = list(CountVectorizer(stop_words="english").get_stop_words()) + [
    "said", "study", "studies", "researchers", "research", "university", "new", "according",
    "percent", "year", "years", "using", "used", "like", "al", "et", "may", "also", "just", "one", "two"]
cv = CountVectorizer(stop_words=VIS_STOP, min_df=5)
Xc = cv.fit_transform(data["clean_text"])
vocab = np.array(cv.get_feature_names_out())
overall = pd.Series(np.asarray(Xc.sum(axis=0)).ravel(), index=vocab).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(x=overall.head(20).values, y=overall.head(20).index, palette="mako", ax=ax)
ax.set_title("Top 20 most frequent words (all six categories)")
ax.set_xlabel("Frequency")
plt.tight_layout()
plt.show()

top_words = {}
for cat in SELECTED:
    idx = np.where(data["Category"].values == cat)[0]
    freq = pd.Series(np.asarray(Xc[idx].sum(axis=0)).ravel(), index=vocab).sort_values(ascending=False)
    top_words[cat] = freq
display(pd.DataFrame({c: list(top_words[c].head(8).index) for c in SELECTED}))

In [ ]:
# Word clouds (saved as images/wordcloud.png).
# If the optional 'wordcloud' package is not installed, a word-frequency panel is produced instead.
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, cat in zip(axes.ravel(), SELECTED):
    freq = top_words[cat].head(60)
    if WORDCLOUD_AVAILABLE:
        wc = WordCloud(width=700, height=450, background_color="white", colormap="viridis",
                       random_state=RANDOM_STATE).generate_from_frequencies(freq.to_dict())
        ax.imshow(wc, interpolation="bilinear")
        ax.axis("off")
    else:
        sub = freq.head(12)[::-1]
        ax.barh(sub.index, sub.values, color="#2E7D32")
        ax.tick_params(axis="y", labelsize=11)
    ax.set_title(cat, fontsize=14)
kind = "Word clouds" if WORDCLOUD_AVAILABLE else "Top-word frequency panels (wordcloud package unavailable)"
fig.suptitle(kind + " by category", fontsize=16)
plt.tight_layout()
fig.savefig(IMAGES_DIR / "wordcloud.png", dpi=130)
plt.show()
print("Image type generated:", "true word cloud" if WORDCLOUD_AVAILABLE else "frequency-panel fallback (wordcloud not installed in this environment)")

## Section 8: TF-IDF Feature Extraction

**Why TF-IDF works for text classification:** Term Frequency (TF) measures how often a word appears in an article, and Inverse Document Frequency (IDF) down-weights words that appear in many articles (they carry little information) while up-weighting words that are distinctive. The result is a numeric vector per article in which words like *ozone*, *emissions* or *crop* get high weights where they are informative. Classical classifiers work well on these sparse vectors.

**Settings used:** English stop words, unigrams + bigrams, `max_features=30000`, `min_df=3`, `sublinear_tf=True`.

**Preventing data leakage:** the vectorizer is part of an sklearn `Pipeline`. During training and cross-validation it is fitted **only on the training portion** of each fold, and the test set is never seen by it.

In [ ]:
def make_pipeline(classifier):
    return Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2),
                                  max_features=30000, min_df=3, sublinear_tf=True)),
        ("clf", classifier),
    ])
print(make_pipeline(MultinomialNB()))

## Section 9: Train-Test Split

80% training / 20% testing, `random_state=42`, stratified by category. The test set stays untouched during model selection.

In [ ]:
X = data["clean_text"]
y = data["Category"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)

print("Total selected records:", len(data))
print("Training records:", len(X_train))
print("Testing records:", len(X_test))

split_df = pd.DataFrame({
    "Train": y_train.value_counts().reindex(SELECTED),
    "Test": y_test.value_counts().reindex(SELECTED)})
split_df["Train %"] = (split_df["Train"] / len(y_train) * 100).round(1)
split_df["Test %"] = (split_df["Test"] / len(y_test) * 100).round(1)
display(split_df)
assert set(y_train) == set(SELECTED) and set(y_test) == set(SELECTED)
print("Verified: all six categories appear in both the training and the testing set.")

## Section 10: Train Machine Learning Models

Three models are compared, each inside a pipeline `TF-IDF -> classifier`:

1. Multinomial Naive Bayes (default settings)
2. Logistic Regression (`C=10`, `max_iter=1000`)
3. Support Vector Classifier (linear SVC, `LinearSVC`) wrapped in a **`CalibratedClassifierCV`** (sigmoid / Platt calibration with 5-fold internal cross-validation) so that it produces genuine, calibrated probability estimates. (The `probability=True` option of `SVC` is deprecated in the installed scikit-learn 1.9 and was very slow here, so the calibrated-classifier route allowed by the project brief is used.)

Hyper-parameters are fixed in advance (no tuning on the test set). Each model is evaluated with **five-fold stratified cross-validation on the training set only**. The primary selection metric is the mean cross-validated **Macro F1-score**; the winner is decided by the actual numbers below, not in advance.

In [ ]:
classifiers = {
    "Multinomial Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, C=10.0, random_state=RANDOM_STATE),
    "Support Vector Classifier (SVC)": CalibratedClassifierCV(
        LinearSVC(C=1.0, random_state=RANDOM_STATE), method="sigmoid", cv=5),
}
pipelines = {name: make_pipeline(clf) for name, clf in classifiers.items()}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"accuracy": "accuracy",
           "precision_macro": "precision_macro",
           "recall_macro": "recall_macro",
           "f1_macro": "f1_macro",
           "f1_weighted": "f1_weighted"}

rows = []
for name, pipe in pipelines.items():
    res = cross_validate(pipe, X_train, y_train, cv=skf, scoring=scoring, n_jobs=1)
    rows.append({
        "Model": name,
        "Accuracy": res["test_accuracy"].mean(),
        "Macro Precision": res["test_precision_macro"].mean(),
        "Macro Recall": res["test_recall_macro"].mean(),
        "Macro F1": res["test_f1_macro"].mean(),
        "Macro F1 Std": res["test_f1_macro"].std(),
        "Weighted F1": res["test_f1_weighted"].mean(),
        "Mean Fit Time (s)": res["fit_time"].mean(),
    })
    print(f"Finished cross-validation: {name}  (mean Macro F1 = {rows[-1]['Macro F1']:.4f})")

## Section 11: Model Comparison

In [ ]:
comparison = (pd.DataFrame(rows).sort_values("Macro F1", ascending=False).reset_index(drop=True))
comparison.insert(0, "Rank", comparison.index + 1)
comparison.round(4).to_csv(REPORTS_DIR / "model_comparison.csv", index=False)
display(comparison.round(4))

best_name = comparison.loc[0, "Model"]
best_cv_f1 = float(comparison.loc[0, "Macro F1"])
print(f"Best model by mean 5-fold CV Macro F1: {best_name} ({best_cv_f1:.4f})")
if len(comparison) > 1:
    gap = best_cv_f1 - float(comparison.loc[1, "Macro F1"])
    print(f"Margin over runner-up ({comparison.loc[1, 'Model']}): {gap:.4f} Macro F1  "
          f"(winner's fold-to-fold std: {comparison.loc[0, 'Macro F1 Std']:.4f})")

metric_cols = ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1", "Weighted F1"]
plot_df = comparison.melt(id_vars="Model", value_vars=metric_cols, var_name="Metric", value_name="Score")
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=plot_df, x="Metric", y="Score", hue="Model", palette="viridis", ax=ax)
lo = max(0.0, plot_df["Score"].min() - 0.05)
ax.set_ylim(lo, 1.0)
ax.set_title("5-fold cross-validation comparison (training set only)")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
fig.savefig(IMAGES_DIR / "model_comparison.png", dpi=150)
plt.show()

**Why the winner was selected:** the model with the highest mean cross-validated Macro F1-score is chosen (printed above). Macro F1 averages the per-class F1 equally, so it rewards a model that performs well on *all six* categories, including the smaller ones such as Environment, rather than only on the large ones. The margin over the runner-up and the fold-to-fold standard deviation are printed above so the reader can judge how clear the difference is. The test set played no part in this decision.

## Section 12: Final Model Evaluation

The selected pipeline is re-trained on the **complete training set** and evaluated **once** on the untouched test set.

The final label for an article is the category with the **highest predicted probability** (the same rule the Streamlit app uses). The notebook also reports how often this agrees with the classifier's own `.predict()` output.

In [ ]:
best_pipe = clone(pipelines[best_name]).fit(X_train, y_train)
classes = np.array(best_pipe.classes_)
proba_test = best_pipe.predict_proba(X_test)
y_pred = classes[proba_test.argmax(axis=1)]
agreement = float((y_pred == best_pipe.predict(X_test)).mean())

test_metrics = {
    "best_model": best_name,
    "selection_metric": "mean 5-fold cross-validation Macro F1 (training set)",
    "cv_macro_f1": best_cv_f1,
    "test_accuracy": float(accuracy_score(y_test, y_pred)),
    "test_macro_precision": float(precision_score(y_test, y_pred, average="macro")),
    "test_macro_recall": float(recall_score(y_test, y_pred, average="macro")),
    "test_macro_f1": float(f1_score(y_test, y_pred, average="macro")),
    "test_weighted_f1": float(f1_score(y_test, y_pred, average="weighted")),
    "prediction_rule": "category with the highest predicted probability",
    "agreement_argmax_proba_vs_predict": agreement,
    "train_records": int(len(X_train)),
    "test_records": int(len(X_test)),
}
with open(REPORTS_DIR / "best_model_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=2)
print(json.dumps(test_metrics, indent=2))

In [ ]:
report = classification_report(y_test, y_pred, labels=SELECTED, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).T
report_df.round(4).to_csv(REPORTS_DIR / "classification_report.csv")
print("Classification report (test set):")
print(classification_report(y_test, y_pred, labels=SELECTED, digits=4, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=SELECTED)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=SELECTED, yticklabels=SELECTED, ax=ax)
ax.set_xlabel("Predicted category")
ax.set_ylabel("Actual category")
ax.set_title(f"Confusion matrix - {best_name} (test set)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
fig.savefig(IMAGES_DIR / "confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
avp = pd.DataFrame({"Actual": y_test.values, "Predicted": y_pred,
                    "Confidence (%)": (proba_test.max(axis=1) * 100).round(1)})
avp["Correct"] = avp["Actual"] == avp["Predicted"]
print("Actual vs predicted - first 15 test articles:")
display(avp.head(15))
print(f"Correct: {int(avp['Correct'].sum())} / {len(avp)}  |  Misclassified: {int((~avp['Correct']).sum())}")
display(avp.groupby("Actual")["Correct"].agg(["sum", "count"]).reindex(SELECTED)
        .rename(columns={"sum": "correct", "count": "test articles"}))
print("Most frequent confusions (actual -> predicted):")
display(avp[~avp["Correct"]].groupby(["Actual", "Predicted"]).size().sort_values(ascending=False).head(8).rename("count").to_frame())

## Section 13: Probability Predictions

Probabilities come from `predict_proba()` of the fitted pipeline (for the SVC, from `CalibratedClassifierCV`, i.e. Platt scaling fitted with internal cross-validation). Nothing is assigned manually, and raw decision scores are never shown as probabilities.

In [ ]:
def predict_probabilities(pipe, text):
    # Returns a DataFrame with all six categories, sorted from highest to lowest probability.
    proba = pipe.predict_proba([clean_text(text)])[0]
    out = pd.DataFrame({"Category": pipe.classes_, "Probability (%)": proba * 100})
    return out.sort_values("Probability (%)", ascending=False).reset_index(drop=True)

# Verify the probability requirements on the whole test set
pct = proba_test * 100
print("Categories per prediction:", proba_test.shape[1], "->", list(best_pipe.classes_))
print("All percentages within [0, 100]:", bool(((pct >= 0) & (pct <= 100)).all()))
print("All row totals approximately 100%:", bool(np.allclose(pct.sum(axis=1), 100.0)))
print("Min / max row total:", round(pct.sum(axis=1).min(), 4), "/", round(pct.sum(axis=1).max(), 4))
print("Final prediction (highest probability) agrees with .predict():", f"{agreement * 100:.2f}% of test articles")
assert set(best_pipe.classes_) == set(SELECTED)

## Section 14: Test New Articles

Original example paragraphs written for this demonstration. They are **not** part of the training or test data. The last example is deliberately unrelated to the six topics to show that the model must still pick one of its six categories.

In [ ]:
demo_articles = [
    ("Climate", "Global average temperatures have continued to rise as atmospheric greenhouse gas concentrations reach record highs. Climate models project that without rapid cuts in carbon dioxide emissions, the planet could warm by more than two degrees Celsius this century, accelerating the loss of Arctic sea ice and raising sea levels."),
    ("Pollution", "Researchers measured high concentrations of fine particulate matter and nitrogen dioxide in the air above several large cities. The study links vehicle exhaust and industrial smoke to respiratory illness, and also found plastic waste and chemical contaminants in nearby rivers and drinking water."),
    ("Environment", "Conservation scientists surveyed a protected rainforest to measure how habitat loss and land use change affect biodiversity. The team recommends restoring wetlands and wildlife corridors so that native ecosystems can recover and endangered species can move between fragmented areas."),
    ("Weather", "A powerful storm system brought heavy rainfall, strong winds and flash flooding to the coast this week. Forecasters say a shift in the jet stream and unusually warm ocean water intensified the cyclone, and meteorologists are tracking another front that may bring thunderstorms and hail."),
    ("Ozone Holes", "Satellite measurements show that the hole in the stratospheric ozone layer over Antarctica is slowly shrinking since the Montreal Protocol phased out chlorofluorocarbons. Scientists are monitoring ozone depletion and ultraviolet radiation levels to confirm that the protective layer continues to recover."),
    ("Agriculture & Food", "A new drought-tolerant wheat variety could help farmers increase crop yields while using less fertilizer and irrigation water. Agronomists tested the variety in several fields and say it may improve food security by protecting harvests of grain, maize and rice from heat stress."),
    ("(unrelated text)", "The football team scored twice in the second half to win the championship final, and the coach praised the players and the supporters in the stadium after the match."),
]
for expected, text in demo_articles:
    result = predict_probabilities(best_pipe, text)
    print("=" * 100)
    print("INPUT:", text[:200] + ("..." if len(text) > 200 else ""))
    print("Intended topic:", expected)
    display(result.assign(**{"Probability (%)": result["Probability (%)"].round(2)}))
    print("FINAL PREDICTION:", result.loc[0, "Category"].upper(), f"({result.loc[0, 'Probability (%)']:.2f}%)")

## Section 15: Save Models

In [ ]:
joblib.dump(best_pipe, MODELS_DIR / "best_model.joblib", compress=3)
(MODELS_DIR / "best_model_name.txt").write_text(best_name + "\n", encoding="utf-8")

svc_name = "Support Vector Classifier (SVC)"
if best_name == svc_name:
    svc_pipe = best_pipe
else:
    svc_pipe = clone(pipelines[svc_name]).fit(X_train, y_train)
joblib.dump(svc_pipe, MODELS_DIR / "svc_model.joblib", compress=3)

# Verify that the saved files reload and reproduce the in-memory predictions
reloaded = joblib.load(MODELS_DIR / "best_model.joblib")
assert np.allclose(reloaded.predict_proba(X_test), proba_test)
reloaded_svc = joblib.load(MODELS_DIR / "svc_model.joblib")
print("Saved best model :", best_name)
for fn in ["best_model.joblib", "best_model_name.txt", "svc_model.joblib"]:
    print(f"  models/{fn}: {(MODELS_DIR / fn).stat().st_size / 1e6:.2f} MB")
print("Reloaded best model reproduces test-set probabilities exactly: True")
print("Pipeline steps:", [s for s, _ in reloaded.steps])

## Section 16: Dataset Statistics

In [ ]:
stats = {
    "original_dataset_records": int(len(df)),
    "original_number_of_categories": int(df["Category"].nunique()),
    "filtered_dataset_records": int(len(df_six)),
    "records_used_after_cleaning": int(len(data)),
    "empty_text_records_removed": int(n_empty_removed),
    "duplicate_articles_removed": int(n_dup_removed),
    "selected_category_counts_filtered": {c: int(actual[c]) for c in SELECTED},
    "selected_category_counts_after_cleaning": {c: int(cat_counts[c]) for c in SELECTED},
    "training_records": int(len(X_train)),
    "testing_records": int(len(X_test)),
    "number_of_selected_categories": len(SELECTED),
    "selected_categories": SELECTED,
}
with open(REPORTS_DIR / "dataset_statistics.json", "w") as f:
    json.dump(stats, f, indent=2)
print(json.dumps(stats, indent=2))

## Section 17: Final Conclusion

In [ ]:
print("FINAL CONCLUSION (generated from the actual execution results above)")
print("-" * 70)
print(f"Dataset      : SciDCC.csv, {len(df):,} original records in {df['Category'].nunique()} categories.")
print(f"Six classes  : {', '.join(SELECTED)}")
print(f"Filtered     : {len(df_six):,} records ({', '.join(f'{c}={actual[c]}' for c in SELECTED)}).")
print(f"After cleaning: {len(data):,} records ({n_empty_removed} empty removed, {n_dup_removed} duplicate articles removed).")
print(f"Split        : {len(X_train):,} training / {len(X_test):,} testing (80/20, stratified, random_state=42).")
print("Features     : TF-IDF (English stop words, unigrams+bigrams, max 30,000 features, min_df=3, sublinear TF) inside sklearn Pipelines.")
print("Models       : " + "; ".join(f"{r['Model']} (CV Macro F1 {r['Macro F1']:.4f})" for _, r in comparison.iterrows()))
print(f"Best model   : {best_name} (selected by mean 5-fold CV Macro F1 = {best_cv_f1:.4f}).")
print(f"Test accuracy: {test_metrics['test_accuracy']:.4f}")
print(f"Test Macro F1: {test_metrics['test_macro_f1']:.4f}   (Weighted F1 {test_metrics['test_weighted_f1']:.4f})")

### Limitations
- The model only knows six categories and **always** outputs one of them, even for unrelated text; the highest probability is not proof that an article truly belongs to that category.
- Categories overlap thematically (for example Climate, Weather and Environment), which limits achievable accuracy; the confusion matrix above shows where mistakes occur.
- The articles come from a single news source (ScienceDaily) and are written in a particular style; performance on other writing styles (social media, blogs, other news outlets) is not measured here.
- Probabilities from classical models are not guaranteed to be perfectly calibrated.
- Hyper-parameters were fixed rather than tuned, and only three classical models were compared.
- Very short inputs contain little information and give unreliable predictions.

### Future improvements
- Tune hyper-parameters with cross-validated search on the training set only.
- Try transformer-based models (for example fine-tuned BERT-style encoders) and compare them with the TF-IDF baselines.
- Add an "Other / not relevant" class or an out-of-domain detector so unrelated text is not forced into one of the six classes.
- Evaluate and calibrate probabilities (for example reliability curves) and test on articles from other sources.
- Add explanations (top contributing words) to the app.